# Hyperparameter Tuning dengan Optuna

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmad-fathan/Deep-Learning-Course/blob/main/hyperparameter-tuning/hyperparameter_tuning_optuna.ipynb)

Versi kedua dari contoh Random Search manual: kini pencariannya diserahkan ke **Optuna**, dengan sampler TPE (Bayesian Optimization) dan pruner Hyperband. Model, data, dan pembagian train/validation/test sama persis, supaya hasilnya bisa dibandingkan.


---
## 1. Instalasi dan impor

Optuna tidak tersedia di Colab secara default, jadi di-install dulu. TensorFlow dan scikit-learn sudah ada.


In [1]:
%pip install -q optuna

import numpy as np
import optuna
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED = 42
tf.random.set_seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

Note: you may need to restart the kernel to use updated packages.


---
## 2. Data: train / validation / test

Sama seperti contoh sebelumnya: validation untuk memilih konfigurasi, test dipakai sekali di akhir.


In [2]:
digits = load_digits()
x, y = digits.data.astype("float32"), digits.target.astype("int32")

x_tr, x_tmp, y_tr, y_tmp = train_test_split(x, y, test_size=0.4, random_state=SEED, stratify=y)
x_va, x_te, y_va, y_te = train_test_split(x_tmp, y_tmp, test_size=0.5, random_state=SEED, stratify=y_tmp)

print(x_tr.shape, x_va.shape, x_te.shape)

(1078, 64) (359, 64) (360, 64)


---
## 3. Objective function

Isi `objective(trial)` sama dengan satu trial pada Random Search, hanya saja konfigurasi tidak lagi diacak sendiri melainkan diminta ke Optuna lewat `trial.suggest_*`.

Tiga hal yang perlu diperhatikan:

- `log=True` pada learning rate menerapkan sampling skala logaritmik (Bagian 4.4 modul).
- Skor yang dikembalikan adalah **akurasi validasi**, bukan test.
- `trial.report()` + `should_prune()` melaporkan akurasi validasi setiap epoch sehingga Hyperband dapat menghentikan trial yang tertinggal (Bagian 5.1–5.2).


In [3]:
N_EPOCHS = 40


def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    units = trial.suggest_int("units", 32, 256, step=32)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    model = tf.keras.Sequential([
        tf.keras.layers.Input((64,)),
        tf.keras.layers.Dense(units, activation="relu"),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(units // 2, activation="relu"),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    best_acc = 0.0
    for epoch in range(N_EPOCHS):
        model.fit(x_tr, y_tr, epochs=1, batch_size=32, verbose=0)
        val_acc = model.evaluate(x_va, y_va, verbose=0)[1]
        best_acc = max(best_acc, val_acc)

        trial.report(val_acc, epoch)
        if trial.should_prune():        # Hyperband menghentikan trial yang tertinggal
            raise optuna.TrialPruned()
    return best_acc

---
## 4. Study: TPE + Hyperband

Sampler default Optuna adalah TPE, salah satu bentuk Bayesian Optimization (Bagian 4.5). Prunernya Hyperband menjalankan Successive Halving berulang kali (Bagian 5.2).


In [4]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.HyperbandPruner(),
)
study.optimize(objective, n_trials=12, timeout=None)

print("jumlah trial selesai :", len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]))
print("jumlah trial dipangkas:", len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]))
print("best params:", study.best_params)
print(f"best val_acc: {study.best_value:.3f}")

jumlah trial selesai : 7
jumlah trial dipangkas: 5
best params: {'lr': 0.0013292918943162175, 'units': 256, 'dropout': 0.36599697090570255}
best val_acc: 0.983


---
## 5. Uji sekali pada test set

Model dilatih ulang memakai konfigurasi terbaik dan seluruh data train+val, lalu diuji pada test set yang belum pernah dipakai saat tuning.


In [5]:
p = study.best_params
x_all, y_all = np.concatenate([x_tr, x_va]), np.concatenate([y_tr, y_va])

tf.random.set_seed(SEED)
model = tf.keras.Sequential([
    tf.keras.layers.Input((64,)),
    tf.keras.layers.Dense(p["units"], activation="relu"),
    tf.keras.layers.Dropout(p["dropout"]),
    tf.keras.layers.Dense(p["units"] // 2, activation="relu"),
    tf.keras.layers.Dropout(p["dropout"]),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(p["lr"]),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_all, y_all, epochs=N_EPOCHS, batch_size=32, verbose=0)

test_loss, test_acc = model.evaluate(x_te, y_te, verbose=0)
print(f"test_acc: {test_acc:.3f}")

test_acc: 0.981


---
## Latihan

1. Bandingkan dengan `hyperparameter_tuning.ipynb`: apakah Optuna menemukan konfigurasi bagus dengan trial lebih sedikit?
2. Ganti sampler menjadi `optuna.samplers.RandomSampler(seed=42)`. Apakah hasilnya membaik atau memburuk?
3. Matikan pruner (`pruner=optuna.pruners.NopPruner()`) dan bandingkan total waktu eksekusi.


In [6]:
# TODO: tulis jawaban Latihan di sini